# Query Builders and SQL Safety

## Overview
SQL injection is the most common and most dangerous database vulnerability. It occurs when user-controlled input is embedded directly into a SQL string, allowing an attacker to change the structure of the query.

**The rule: never concatenate user input into SQL.**

Every value that originates outside your code (HTTP request parameters, form fields, file contents, API responses) must be passed as a **parameter**, not embedded as a string.

**The three safe patterns in Python:**
```python
# 1. sqlite3 / psycopg2: positional ? or %s
conn.execute("SELECT * FROM users WHERE name = ?", (name,))
conn.execute("SELECT * FROM users WHERE name = %s", (name,))  # psycopg2

# 2. SQLAlchemy text() with named parameters
conn.execute(text("SELECT * FROM users WHERE name = :name"), {"name": name})

# 3. SQLAlchemy Core expression (never touches strings)
select(users).where(users.c.name == name)
```

**What cannot be parameterised:** table names and column names. These must be validated against an **allowlist** before being embedded in SQL.

---

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
conn.executescript("""
    CREATE TABLE users (user_id INTEGER PRIMARY KEY, username TEXT, password_hash TEXT, role TEXT);
    INSERT INTO users VALUES (1,'admin','hash_abc','admin'),(2,'alice','hash_xyz','user');
""")
conn.commit()

print('=== SQL injection: the attack ===')

def unsafe_login(conn, username, password):
    """NEVER do this -- classic SQL injection."""
    query = f"SELECT * FROM users WHERE username = '{username}' AND password_hash = '{password}'"
    print(f'  Query: {query}')
    return conn.execute(query).fetchall()

# Normal usage
result = unsafe_login(conn, 'alice', 'hash_xyz')
print(f'Normal login: {[dict(r) for r in result]}')

# SQL injection: bypass password check entirely
result = unsafe_login(conn, "admin' --", 'anything')
print(f'Injection login (bypassed password!): {[dict(r) for r in result]}')
# The query becomes:
# SELECT * FROM users WHERE username = 'admin' --' AND password_hash = 'anything'
# Everything after -- is a comment; the password check is gone.

# Another attack: drop table
# username = "'; DROP TABLE users; --"
# query = SELECT * FROM users WHERE username = ''; DROP TABLE users; --'

print()
print('=== Parameterised query: the fix ===')

def safe_login(conn, username, password):
    """Correct: parameterised query."""
    return conn.execute(
        'SELECT * FROM users WHERE username = ? AND password_hash = ?',
        (username, password)
    ).fetchall()

result = safe_login(conn, "admin' --", 'anything')
print(f'Injection attempt with safe query: {result}')  # returns [] -- no match
print('The injection string is treated as a literal value, not SQL syntax.')


---
## Safe dynamic query building with SQLAlchemy Core

In [ ]:
from sqlalchemy import create_engine, text, select, and_, or_
from sqlalchemy import MetaData
import warnings; warnings.filterwarnings('ignore')

engine = create_engine('sqlite:///:memory:')
conn_s = engine.connect()
conn_s.execute(text(
    'CREATE TABLE employees (emp_id INTEGER PRIMARY KEY, full_name TEXT, dept_id INTEGER, salary REAL, is_active INTEGER DEFAULT 1)'
))
conn_s.execute(text(
    "INSERT INTO employees VALUES (1,'Aroha',2,95000,1),(2,'Liam',1,110000,1),(3,'Fatima',2,88000,1),(4,'James',3,75000,1)"
))
conn_s.commit()

meta = MetaData()
meta.reflect(bind=engine)
employees = meta.tables['employees']

print('=== Safe dynamic query building with SQLAlchemy Core ===')

def search_employees(dept_id=None, min_salary=None, active_only=True):
    stmt = select(
        employees.c.emp_id,
        employees.c.full_name,
        employees.c.salary,
    )
    # Build WHERE clauses dynamically — no string concatenation
    conditions = []
    if active_only:
        conditions.append(employees.c.is_active == 1)
    if dept_id is not None:
        conditions.append(employees.c.dept_id == dept_id)
    if min_salary is not None:
        conditions.append(employees.c.salary >= min_salary)
    if conditions:
        stmt = stmt.where(and_(*conditions))
    stmt = stmt.order_by(employees.c.salary.desc())
    with engine.connect() as conn:
        return conn.execute(stmt).fetchall()

print('All active (no filters):')
for r in search_employees():
    print(f'  {r.full_name:<14s}  ${r.salary:,.0f}')

print('\nDept 2, min salary $90k:')
for r in search_employees(dept_id=2, min_salary=90000):
    print(f'  {r.full_name:<14s}  ${r.salary:,.0f}')


---
## Allowlists for dynamic table/column names

In [ ]:
print('=== Allowlist for dynamic table/column names ===')

# Table and column names CANNOT be parameterised with ? or :name
# They must be embedded in the SQL string.
# NEVER embed raw user input directly -- use an allowlist.

ALLOWED_SORT_COLS = {'emp_id', 'full_name', 'salary', 'hire_date'}
ALLOWED_TABLES    = {'employees', 'departments', 'projects'}

def safe_sorted_query(conn, sort_col, sort_dir='asc'):
    # Validate against allowlist before embedding in SQL
    if sort_col not in ALLOWED_SORT_COLS:
        raise ValueError(f'Invalid sort column: {sort_col!r}')
    if sort_dir.lower() not in ('asc', 'desc'):
        raise ValueError(f'Invalid sort direction: {sort_dir!r}')
    # Safe to embed now — value comes from allowlist, not raw user input
    query = f'SELECT emp_id, full_name, salary FROM employees ORDER BY {sort_col} {sort_dir.upper()}'
    return conn.execute(text(query)).fetchall()

with engine.connect() as conn:
    print('Sorted by salary DESC:')
    for r in safe_sorted_query(conn, 'salary', 'desc'):
        print(f'  emp_id={r.emp_id}  {r.full_name:<14s}  ${r.salary:,.0f}')

    print('\nAllowlist rejection:')
    try:
        safe_sorted_query(conn, 'password_hash', 'asc')  # not in allowlist
    except ValueError as e:
        print(f'  Rejected: {e}')

print()
print('=== Summary: safe SQL in Python ===')
rules = [
    ('ALWAYS use ?/:name for values',     'Never f-string user input into SQL'),
    ('ALWAYS allowlist table/col names',   'They cannot be parameterised; validate first'),
    ('Use SQLAlchemy Core for dynamic SQL','and_() / or_() / select() avoid string concat'),
    ('Least-privilege DB credentials',     'App user: SELECT+DML only; no DROP/CREATE'),
    ('Audit input validation',             'Validate input type, range, and length before SQL'),
    ('Use an ORM or query builder',        'They parameterise by default; harder to misuse'),
]
for rule, reason in rules:
    print(f'  {rule:<40s}  {reason}')


---
## Common Pitfalls

**1. Any f-string or `%` formatting with user-controlled values in SQL**
This is the definition of SQL injection. `f"WHERE name = '{user_input}'"` is exploitable regardless of how the rest of the application is built. There are no exceptions. Every value must go through a parameterised placeholder: `?` (sqlite3/psycopg2), `:name` (SQLAlchemy `text()`), or a SQLAlchemy expression object.

**2. Parameterising table and column names the wrong way**
`conn.execute('SELECT ? FROM employees', ('salary',))` does not work -- the `?` is treated as a string value and the query returns the literal string 'salary', not the column data. Identifiers (table names, column names) cannot be parameterised. They must come from an explicit allowlist in application code.

**3. Trusting ORM to prevent all injection**
SQLAlchemy ORM and Core are safe when used correctly. But `conn.execute(text(f'... {user_input} ...'))` is just as vulnerable as raw sqlite3 with string concatenation. The f-string bypass is the risk, not the library. Always use named parameters in `text()` calls.

**4. Logging SQL queries including parameter values in production**
`echo=True` or `echo_pool=True` logs every SQL statement. If the log includes parameter values (passwords, PII, card numbers), those values appear in plaintext in log files. Use `echo=False` in production and configure parameterised log redaction if query logging is needed.

**5. Building OR conditions with string concatenation instead of or_()**
Building `WHERE col IN (...)` with a loop that concatenates values into a string is both an injection risk and ugly. Use SQLAlchemy's `col.in_(list_of_values)` which correctly generates `WHERE col IN (?, ?, ?)` with all values parameterised.

**6. Overly broad database credentials**
An application's database user should have only the permissions it needs: `SELECT`, `INSERT`, `UPDATE`, `DELETE` on the specific tables it uses. No `DROP TABLE`, `CREATE TABLE`, `TRUNCATE`, or superuser rights. If an injection attack succeeds, least-privilege credentials limit the blast radius to the rows the application can touch.


---
*sql_methods_library - Samantha McGarrigle*